In [3]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

df = pd.read_csv('../data/cleaned_nhl_data.csv')

df = df[df['games_played'] >= 15]

features = ['PTS', 'Age', 'games_played', 'icetime', 'I_F_xGoals', 'points_per_game']
X = df[features]
y = df['AAV']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ridge_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', Ridge())
])

rf_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(random_state=42))
])

param_grid_ridge = {'model__alpha': [0.1, 1.0, 10.0, 100.0]}
param_grid_rf = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5]
}

print("Starting GridSearch... (this may take a moment)")
grid_ridge = GridSearchCV(ridge_pipe, param_grid_ridge, cv=5, scoring='r2')
grid_rf = GridSearchCV(rf_pipe, param_grid_rf, cv=5, scoring='r2')

grid_ridge.fit(X_train, y_train)
grid_rf.fit(X_train, y_train)

def evaluate_model(grid_obj, name):
    preds = grid_obj.predict(X_test)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    print(f"\n--- {name} Results ---")
    print(f"Best Params: {grid_obj.best_params_}")
    print(f"R2 Score: {r2:.4f}")
    print(f"MAE: ${mae:,.2f}")
    print(f"RMSE: ${rmse:,.2f}")
    return r2, mae

ridge_r2, ridge_mae = evaluate_model(grid_ridge, "Ridge Regression")
rf_r2, rf_mae = evaluate_model(grid_rf, "Random Forest")

print("\nGenerating SHAP Summary Plot...")
best_rf_model = grid_rf.best_estimator_.named_steps['model']
preprocessor = Pipeline(grid_rf.best_estimator_.steps[:-1])
X_test_transformed = preprocessor.transform(X_test)

explainer = shap.TreeExplainer(best_rf_model)
shap_values = explainer.shap_values(X_test_transformed)

shap.summary_plot(shap_values, X_test_transformed, feature_names=features)

ModuleNotFoundError: No module named 'shap'

In [ ]:
all_preds = grid_rf.predict(X)

results = df[['name', 'AAV']].copy()
results['Predicted_AAV'] = all_preds
results['Difference'] = results['Predicted_AAV'] - results['AAV']

underpaid = results.sort_values(by='Difference', ascending=False).head(10)

overpaid = results.sort_values(by='Difference', ascending=True).head(10)

print("--- Top 10 Undervalued Players (Model Value > Actual Pay) ---")
print(underpaid[['name', 'AAV', 'Predicted_AAV', 'Difference']].head(10))

print("\n--- Top 10 Overvalued Players (Actual Pay > Model Value) ---")
print(overpaid[['name', 'AAV', 'Predicted_AAV', 'Difference']].head(10))

--- Top 10 Undervalued Players (Model Value > Actual Pay) ---
                     name        AAV  Predicted_AAV    Difference
466           Kyle Connor  7142857.0   1.033594e+07  3.193084e+06
51        Jackson LaCombe   925000.0   3.876211e+06  2.951211e+06
362            Ryan Suter   775000.0   3.713064e+06  2.938064e+06
729      Marcus Johansson   800000.0   3.726781e+06  2.926781e+06
64        Simon Edvinsson   894167.0   3.719360e+06  2.825193e+06
370         Stefan Noesen  2750000.0   5.475845e+06  2.725845e+06
496  Oliver Ekman-Larsson  3500000.0   6.186229e+06  2.686229e+06
377           Brent Burns  1000000.0   3.409228e+06  2.409228e+06
260        Josh Morrissey  6250000.0   8.622376e+06  2.372376e+06
263    Mackie Samoskevich   775000.0   3.095853e+06  2.320853e+06

--- Top 10 Overvalued Players (Actual Pay > Model Value) ---
                 name         AAV  Predicted_AAV    Difference
171   Hampus Lindholm   6500000.0   1.275337e+06 -5.224663e+06
774      Tyler Seguin   

In [ ]:
def check_player_value(player_name):
    player_data = results[results['name'].str.contains(player_name, case=False, na=False)]
    
    if player_data.empty:
        return f"Player '{player_name}' not found in the dataset."
    
    row = player_data.iloc[0]
    status = "UNDERVALUED" if row['Difference'] > 0 else "OVERVALUED"
    
    print(f"--- Analysis for {row['name']} ---")
    print(f"Actual AAV:    ${row['AAV']:,.0f}")
    print(f"Predicted AAV: ${row['Predicted_AAV']:,.0f}")
    print(f"Difference:    ${row['Difference']:,.0f} ({status})")

check_player_value("Leon Draisaitl")
print("\n")
check_player_value("Kirill Kaprizov")

--- Analysis for Leon Draisaitl ---
Actual AAV:    $14,000,000
Predicted AAV: $12,192,910
Difference:    $-1,807,090 (OVERVALUED)


--- Analysis for Kirill Kaprizov ---
Actual AAV:    $9,000,000
Predicted AAV: $7,476,497
Difference:    $-1,523,503 (OVERVALUED)


In [ ]:
import joblib

joblib.dump(grid_rf.best_estimator_, '../src/rf_model.pkl')

results = df[['name', 'AAV', 'PTS', 'I_F_xGoals', 'Age']].copy()
results['Predicted_AAV'] = all_preds
results['Difference'] = results['Predicted_AAV'] - results['AAV']

results.to_csv('../data/valuation_results.csv', index=False)

print("Model and results exported successfully!")

Model and results exported successfully!
